In [ ]:
import os
import re
from collections import Counter

import pandas as pd

import cleaning_pipeline as cp
from data_loader import load_imdb_data

# Load the raw data
df = load_imdb_data()

# Recreate the output of Member 1's Steps 2-4 with the same functions (from cleaning_pipeline.py)
df_processed = df.copy()
df_processed["normalized_review"] = df_processed["review"].apply(cp.normalize_unicode)   # Step 2
df_processed["cleaned_review"] = df_processed["normalized_review"].apply(cp.clean_text)   # Step 3
df_processed["regex_review"] = df_processed["cleaned_review"].apply(cp.regex_process)     # Step 4

rating_pattern = r"\b\d{1,2}/10\b"   # rating pattern from Step 4

import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

df_processed["lowercase_review"] = df_processed["regex_review"].str.lower()                 # Step 5
df_processed["tokens"] = df_processed["lowercase_review"].apply(cp.tokenize_review)         # Step 6
df_processed["tokens_no_stopwords"] = df_processed["tokens"].apply(cp.remove_stopwords)     # Step 7

# Step 8: stemming (same code as Member 2)
porter_stemmer = PorterStemmer()
unique_tokens_no_stopwords = set()
for tokens in df_processed["tokens_no_stopwords"]:
    unique_tokens_no_stopwords.update(tokens)
stem_lookup = {word: porter_stemmer.stem(word) for word in unique_tokens_no_stopwords}
df_processed["tokens_stemmed"] = df_processed["tokens_no_stopwords"].apply(lambda tokens: [stem_lookup[t] for t in tokens])

print("Shape:", df_processed.shape)
print("Columns:", df_processed.columns.tolist())

## Step 9: Lemmatization

### 9.1 Download the required NLTK resources and create the lemmatizer and POS tagger

In [73]:
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tag import PerceptronTagger

wordnet_lemmatizer = WordNetLemmatizer()
pos_tagger = PerceptronTagger()   # load the POS model only once

### 9.2 Lemmatization with and without POS

In [74]:
demo_words = [
    ("movies", "n"), ("studies", "n"), ("feet", "n"), ("children", "n"),
    ("watched", "v"), ("acting", "v"), ("loved", "v"), ("was", "v"), ("running", "v"),
    ("better", "a"), ("worst", "a"),
]

print(f"{'word':10} {'POS':4} {'without POS':12} {'with POS':10} {'stem (Step 8)'}")
print("-" * 55)
for word, pos in demo_words:
    without_pos = wordnet_lemmatizer.lemmatize(word)
    with_pos = wordnet_lemmatizer.lemmatize(word, pos)
    stem = porter_stemmer.stem(word)
    print(f"{word:10} {pos:4} {without_pos:12} {with_pos:10} {stem}")

word       POS  without POS  with POS   stem (Step 8)
-------------------------------------------------------
movies     n    movie        movie      movi
studies    n    study        study      studi
feet       n    foot         foot       feet
children   n    child        child      children
watched    v    watched      watch      watch
acting     v    acting       act        act
loved      v    loved        love       love
was        v    wa           be         wa
running    v    running      run        run
better     a    better       good       better
worst      a    worst        bad        worst


### 9.3 Convert POS tags into WordNet POS

In [75]:
def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN


example_tokens = df_processed.loc[0, "tokens_no_stopwords"][:12]
print("Tokens:     ", example_tokens)
print("POS tags:   ", pos_tagger.tag(example_tokens))

Tokens:      ['one', 'reviewers', 'mentioned', 'watching', '1', 'oz', 'episode', "'ll", 'hooked', '.', 'right', ',']
POS tags:    [('one', 'CD'), ('reviewers', 'NNS'), ('mentioned', 'VBD'), ('watching', 'VBG'), ('1', 'CD'), ('oz', 'JJ'), ('episode', 'NN'), ("'ll", 'MD'), ('hooked', 'VB'), ('.', '.'), ('right', 'NN'), (',', ',')]


### 9.4 Apply lemmatization to the first 5,000 reviews

In [76]:
import time

# POS tagging is slow, so in the notebook we lemmatize only the first 5,000 reviews
# (cleaning_pipeline.py lemmatizes all 50,000 and saves the result)
lemma_sample_size = 5000

lemma_lookup = {}


def lemmatize_tokens(tokens):
    lemmas = []
    for word, tag in pos_tagger.tag(tokens):
        pos = get_wordnet_pos(tag)
        if (word, pos) not in lemma_lookup:
            lemma_lookup[(word, pos)] = wordnet_lemmatizer.lemmatize(word, pos)
        lemmas.append(lemma_lookup[(word, pos)])
    return lemmas


start_time = time.time()

# Input is the Step 7 column (tokens_no_stopwords), NOT the stemmed tokens
df_processed["tokens_lemmatized"] = df_processed["tokens_no_stopwords"].head(lemma_sample_size).apply(lemmatize_tokens)

print(f"Lemmatized {lemma_sample_size} reviews in {time.time() - start_time:.1f} seconds")
print("Reviews with lemmatized tokens:", df_processed["tokens_lemmatized"].notna().sum())

df_processed[["tokens_no_stopwords", "tokens_lemmatized"]].head()

Lemmatized 5000 reviews in 101.7 seconds
Reviews with lemmatized tokens: 5000


,tokens_no_stopwords,tokens_lemmatized
0,"[one, reviewers, mentioned, watching, 1, oz, e...","[one, reviewer, mention, watch, 1, oz, episode..."
1,"[wonderful, little, production, ., filming, te...","[wonderful, little, production, ., film, techn..."
2,"[thought, wonderful, way, spend, time, hot, su...","[think, wonderful, way, spend, time, hot, summ..."
3,"[basically, 's, family, little, boy, (, jake, ...","[basically, 's, family, little, boy, (, jake, ..."
4,"[petter, mattei, 's, ``, love, time, money, ''...","[petter, mattei, 's, ``, love, time, money, ''..."


### 9.5 Before-and-after examples

In [77]:
def show_lemmatization(index, n_tokens=25):
    before = df_processed.loc[index, "tokens_no_stopwords"][:n_tokens]
    lemmas = df_processed.loc[index, "tokens_lemmatized"][:n_tokens]
    stems = df_processed.loc[index, "tokens_stemmed"][:n_tokens]
    tags = pos_tagger.tag(before)
    print("Review index:", index)
    print("BEFORE (stopwords removed):", before)
    print()
    print("AFTER (lemmatized):        ", lemmas)
    print()
    print(f"{'word':15} {'POS':5} {'lemma':15} {'stem (Step 8)'}")
    for (word, tag), lemma, stem in zip(tags, lemmas, stems):
        if word != lemma or word != stem:
            print(f"{word:15} {tag:5} {lemma:15} {stem}")
    print("-" * 80)


show_lemmatization(0)
show_lemmatization(1)

Review index: 0
BEFORE (stopwords removed): ['one', 'reviewers', 'mentioned', 'watching', '1', 'oz', 'episode', "'ll", 'hooked', '.', 'right', ',', 'exactly', 'happened', '.', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scenes', 'violence', ',', 'set']

AFTER (lemmatized):         ['one', 'reviewer', 'mention', 'watch', '1', 'oz', 'episode', "'ll", 'hook', '.', 'right', ',', 'exactly', 'happen', '.', 'first', 'thing', 'strike', 'oz', 'brutality', 'unflinching', 'scene', 'violence', ',', 'set']

word            POS   lemma           stem (Step 8)
reviewers       NNS   reviewer        review
mentioned       VBD   mention         mention
watching        VBG   watch           watch
episode         NN    episode         episod
hooked          VB    hook            hook
exactly         RB    exactly         exactli
happened        VBD   happen          happen
struck          VBD   strike          struck
brutality       NN    brutality       brutal
unflinching     VBG   unf

### 9.6 Statistics: compare the vocabulary sizes

In [78]:
sample = df_processed.head(lemma_sample_size)


def count_unique(column):
    unique = set()
    for tokens in sample[column]:
        unique.update(tokens)
    return len(unique)


print(f"Unique tokens in the first {lemma_sample_size} reviews:")
print("  After stopword removal (Step 7):", count_unique("tokens_no_stopwords"))
print("  After stemming (Step 8):        ", count_unique("tokens_stemmed"))
print("  After lemmatization (Step 9):   ", count_unique("tokens_lemmatized"))

Unique tokens in the first 5000 reviews:
  After stopword removal (Step 7): 46104
  After stemming (Step 8):         33799
  After lemmatization (Step 9):    40341


### 9.7 Check the results (first 5,000 reviews)

In [79]:
sample = df_processed.head(lemma_sample_size)

length_mismatch = (
    sample["tokens_no_stopwords"].apply(len) != sample["tokens_lemmatized"].apply(len)
).sum()
print(f"Checking the first {lemma_sample_size} reviews")
print("Reviews where the number of tokens changed:", length_mismatch)

print("\nNegation words (Step 7 -> Step 9):")
for word in ["not", "no", "never", "n't"]:
    before = sample["tokens_no_stopwords"].apply(lambda tokens: tokens.count(word)).sum()
    after = sample["tokens_lemmatized"].apply(lambda tokens: tokens.count(word)).sum()
    print(f"  {word:6} {before} -> {after}")

print("\nTokens in Step 7 column (all reviews, unchanged):", df_processed["tokens_no_stopwords"].apply(len).sum())
print("Tokens in Step 8 column (all reviews, unchanged):", df_processed["tokens_stemmed"].apply(len).sum())

print("\nOriginal review column unchanged:", df["review"].equals(df_processed["review"]))
print("df_processed columns:", df_processed.columns.tolist())

Checking the first 5000 reviews
Reviews where the number of tokens changed: 0

Negation words (Step 7 -> Step 9):
  not    6165 -> 6165
  no     2462 -> 2463
  never  1342 -> 1342
  n't    6583 -> 6583

Tokens in Step 7 column (all reviews, unchanged): 7765868
Tokens in Step 8 column (all reviews, unchanged): 7765868

Original review column unchanged: True
df_processed columns: ['review', 'sentiment', 'normalized_review', 'cleaned_review', 'regex_review', 'lowercase_review', 'tokens', 'tokens_no_stopwords', 'tokens_stemmed', 'tokens_lemmatized']


## Step 10: Stemming vs Lemmatization

In [80]:
comparison_words = [
    "movies", "studies", "stories", "happiness", "watched", "better",
    "organization", "organ", "university", "universe", "general", "generous",
]

print(f"{'word':14} {'stem (Step 8)':15} {'lemma (Step 9)'}")
print("-" * 45)
for word in comparison_words:
    tag = pos_tagger.tag([word])[0][1]
    lemma = wordnet_lemmatizer.lemmatize(word, get_wordnet_pos(tag))
    print(f"{word:14} {porter_stemmer.stem(word):15} {lemma}")

word           stem (Step 8)   lemma (Step 9)
---------------------------------------------
movies         movi            movie
studies        studi           study
stories        stori           story
happiness      happi           happiness
watched        watch           watch
better         better          well
organization   organ           organization
organ          organ           organ
university     univers         university
universe       univers         universe
general        gener           general
generous       gener           generous


## Step 11: Single Cleaning Pipeline

### 11.1 Clean one review with clean_review()

In [81]:
import cleaning_pipeline

example_review = df.loc[1, "review"]
print("RAW REVIEW:")
print(example_review[:300])
print()
print("clean_review() OUTPUT (first 40 tokens):")
print(cleaning_pipeline.clean_review(example_review)[:40])

RAW REVIEW:
A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" 

clean_review() OUTPUT (first 40 tokens):
['wonderful', 'little', 'production', '.', 'film', 'technique', 'unassuming', 'old-time-bbc', 'fashion', 'give', 'comfort', ',', 'sometimes', 'discomforting', ',', 'sense', 'realism', 'entire', 'piece', '.', 'actor', 'extremely', 'well', 'choose', 'michael', 'sheen', 'not', '``', 'get', 'polari', "''", 'voice', 'pat', '!', 'truly', 'see', 'seamless', 'edit', 'guided', 'reference']


### 11.2 Check: does the single function give the same result as the step-by-step notebook?

In [82]:
check_size = 200

pipeline_output = df["review"].head(check_size).apply(cleaning_pipeline.clean_review)
step_by_step_output = df_processed["tokens_lemmatized"].head(check_size)

matches = sum(a == b for a, b in zip(pipeline_output, step_by_step_output))
print(f"Reviews where clean_review() matches the step-by-step result: {matches} / {check_size}")

Reviews where clean_review() matches the step-by-step result: 200 / 200


### 11.3 Vocabulary size BEFORE vs AFTER cleaning

In [83]:
first_reviews = df["review"].head(lemma_sample_size)

raw_vocabulary = set()
for review in first_reviews:
    raw_vocabulary.update(word_tokenize(review))


def vocabulary_of(column):
    vocabulary = set()
    for tokens in df_processed[column].head(lemma_sample_size):
        vocabulary.update(tokens)
    return len(vocabulary)


vocabulary_table = pd.DataFrame({
    "Stage": [
        "Raw text (no cleaning)",
        "After Unicode + noise + regex + lowercase + tokenization (Step 6)",
        "After stop-word removal (Step 7)",
        "After lemmatization = final pipeline output (Step 9)",
    ],
    "Vocabulary size": [
        len(raw_vocabulary),
        vocabulary_of("tokens"),
        vocabulary_of("tokens_no_stopwords"),
        vocabulary_of("tokens_lemmatized"),
    ],
})

before = vocabulary_table["Vocabulary size"].iloc[0]
after = vocabulary_table["Vocabulary size"].iloc[-1]
print(vocabulary_table.to_string(index=False))
print(f"\nVocabulary BEFORE cleaning: {before}")
print(f"Vocabulary AFTER cleaning:  {after}")
print(f"Reduction: {(1 - after / before) * 100:.1f}%")

                                                            Stage  Vocabulary size
                                           Raw text (no cleaning)            56910
After Unicode + noise + regex + lowercase + tokenization (Step 6)            46234
                                 After stop-word removal (Step 7)            46104
             After lemmatization = final pipeline output (Step 9)            40341

Vocabulary BEFORE cleaning: 56910
Vocabulary AFTER cleaning:  40341
Reduction: 29.1%


## Step 12: Save Cleaned Dataset

### 12.0 Create the cleaned dataset with the script
Runs `cleaning_pipeline.py` on all 50,000 reviews and saves `data/cleaned_imdb_reviews.csv` (about 20 minutes). Skip this cell if the file already exists.

In [ ]:
import sys

!{sys.executable} -u cleaning_pipeline.py

### 12.1 Load the saved cleaned dataset

In [84]:
cleaned_path = os.path.join("data", "cleaned_imdb_reviews.csv")

if os.path.exists(cleaned_path):
    cleaned_df = pd.read_csv(cleaned_path, keep_default_na=False)
    print("Shape (rows, columns):", cleaned_df.shape)
    print("Columns:", cleaned_df.columns.tolist())
    print("\nSentiment distribution:")
    print(cleaned_df["sentiment"].value_counts())
    print("\nFirst cleaned review (first 300 characters):")
    print(cleaned_df.loc[0, "cleaned_text"][:300])
    print("\nTokens again with .split() (first 20):", cleaned_df.loc[0, "cleaned_text"].split()[:20])
else:
    print("Cleaned file not found yet.")
    print("Run this in a terminal from the project folder:  python cleaning_pipeline.py")

Cleaned file not found yet.
Run this in a terminal from the project folder:  python cleaning_pipeline.py
